This Python project is designed to extract football data from the FBref website. It uses the pandas library to parse HTML tables from match pages and process the data into a structured format.

The project iterates through a list of match URLs, extracts relevant information about each match, with focus on player performance statistics, and saves the extracted data into a CSV file.

The project includes a `time.sleep(10)` call within the loop to introduce a delay between requests to the FBref server. This is crucial to avoid overwhelming the server with too many requests in a short period, which could lead to temporary blocking or rate limiting. It is strongly recommended not to remove or modify this sleep function without careful consideration of the potential consequences.

This project is for educational and personal use only. FBref has its own terms of service and usage policies that should be respected. Please use this project responsibly and ethically.

## Libraries

In [ ]:
import pandas as pd
import re
import time
from google.colab import drive

## Variables

In [ ]:
# Match_ID for the first match processed
# If match_ID = 0, there will be no 'MD' column in the data frame
match_ID = 1

# If MD = 0, there will be no 'MD' column in the data frame
MD = 1

# matches_per_MD will only be taken in consideration if MD > 0
matches_per_MD = 18

merge_keys = ['Player', '#', 'Nation', 'Pos', 'Age', 'Min']

all_data = pd.DataFrame()

# Information about the file that will be created with the data extracted
file_name = 'all_data.csv'
folder_path = '/content/drive/MyDrive/'

file_path = folder_path + file_name

In [ ]:
urls = ['https://fbref.com/en/matches/568274fc/Young-Boys-Aston-Villa-September-17-2024-Champions-League',
 'https://fbref.com/en/matches/931cd5fb/Juventus-PSV-Eindhoven-September-17-2024-Champions-League',
 'https://fbref.com/en/matches/aaca158d/Sporting-CP-Lille-September-17-2024-Champions-League',
 'https://fbref.com/en/matches/4fb5fcbd/Real-Madrid-Stuttgart-September-17-2024-Champions-League',
 'https://fbref.com/en/matches/54ed5557/Milan-Liverpool-September-17-2024-Champions-League',
 'https://fbref.com/en/matches/7c5c2955/Bayern-Munich-Dinamo-Zagreb-September-17-2024-Champions-League',
 'https://fbref.com/en/matches/936f3ee0/Sparta-Prague-Red-Bull-Salzburg-September-18-2024-Champions-League',
 'https://fbref.com/en/matches/85075de5/Bologna-Shakhtar-Donetsk-September-18-2024-Champions-League',
 'https://fbref.com/en/matches/16af036c/Celtic-Slovan-Bratislava-September-18-2024-Champions-League',
 'https://fbref.com/en/matches/db400f4b/Manchester-City-Internazionale-September-18-2024-Champions-League',
 'https://fbref.com/en/matches/722df76c/Club-Brugge-Dortmund-September-18-2024-Champions-League',
 'https://fbref.com/en/matches/53cfba8e/Paris-Saint-Germain-Girona-September-18-2024-Champions-League',
 'https://fbref.com/en/matches/2d931c93/Red-Star-Belgrade-Benfica-September-19-2024-Champions-League',
 'https://fbref.com/en/matches/24c1d39c/Feyenoord-Bayer-Leverkusen-September-19-2024-Champions-League',
 'https://fbref.com/en/matches/745d19c2/Monaco-Barcelona-September-19-2024-Champions-League',
 'https://fbref.com/en/matches/108b9e41/Atalanta-Arsenal-September-19-2024-Champions-League',
 'https://fbref.com/en/matches/162f5233/Brest-Sturm-Graz-September-19-2024-Champions-League',
 'https://fbref.com/en/matches/a112dcca/Atletico-Madrid-RB-Leipzig-September-19-2024-Champions-League',
]






## Functions

In [ ]:
def extract_squads_dfs(dataframe):
  """Extracts the DataFrames containing the squads' data.

  Args:
    dataframe: A list of DataFrames read from the HTML page.

  Returns:
    A tuple containing two lists of DataFrames: (home_squad_dfs, away_squad_dfs).
  """
  # Select DataFrames for home team players (df[3] to df[8])
  # These specific ranges are chosen based on the consistent structure of the HTML page
  # DataFrames 3-8 contain home team data and DataFrames 10-15 contain away team data
  home_dfs = dataframe[3:9]

  # Select DataFrames for away team players (df[10] to df[15])
  away_dfs = dataframe[10:16]

  squads_dfs = home_dfs, away_dfs

  return squads_dfs

In [ ]:
def extract_squad_names(dfs):
  """Extracts the home and away team names from the list of DataFrames.

  Args:
    dfs: The list of DataFrames read from the HTML page.

  Returns:
    A tuple containing (home_name, away_name).
  """
  home_name = dfs[2].columns.tolist()[0][0]
  away_name = dfs[2].columns.tolist()[1][0]
  return home_name, away_name

In [ ]:
def extract_last_word(column_name):
    """Extracts the last word from a column name.

    Args:
        column_name: The column name to extract the last word from.

    Returns:
        The last word of the column name.
    """
    last_word = re.sub(r'.*_', '', column_name)
    return last_word

In [ ]:
def clean_column_names(squad_dfs):
  """Cleans the column names of all DataFrames in a squad list.

  Args:
    squad_dfs: A list of DataFrames representing a squad.

  Returns:
    The list of DataFrames with cleaned column names.
  """
  for df in squad_dfs:
    # Joining the levels with an underscore
    df.columns = ['_'.join(col).strip() for col in df.columns.values]

    # Cleaning column names, removing "Unnamed"
    df.columns = [extract_last_word(col) if 'Unnamed' in col else col for col in df.columns]

  return squad_dfs

In [ ]:
def merge_dfs(dataframes, merge_keys):
  """Merges data for a single team from a list of DataFrames.

  Args:
    dataframes: A list of DataFrames containing data for the team.
    merge_keys: A list of column names to use as merge keys.

  Returns:
    A merged DataFrame containing all the data for the team.
  """
  # Initialize the merged DataFrame with the first DataFrame in the list
  merged_df = dataframes[0]

  # Iterate over the remaining DataFrames in the list
  for df in dataframes[1:]:
    # Select the columns to use for the merge
    # We want to include columns that are not yet present in the merged DataFrame or columns that are in the merge_keys list
    cols_to_use = [col for col in df.columns if col not in merged_df.columns or col in merge_keys]

    # Merge the current DataFrame with the merged DataFrame.
    merged_df = pd.merge(merged_df, df[cols_to_use], on=merge_keys, how='outer')

  # Return the merged DataFrame.
  return merged_df

In [ ]:
def clean_df(dataframe):
  """Cleans and preprocesses a DataFrame containing squad data.

  Args:
    dataframe: A pandas DataFrame containing the raw squad data.

  Returns:
    A cleaned and preprocessed pandas DataFrame.
  """
  # Eliminating rows containing a number followed by 'players' in the 'Player' column. By doing this, we remove the row that aggregates data for all players who played for the team
  # The regex below matches rows with player summaries like '11 Players'
  dataframe = dataframe[~dataframe['Player'].str.contains(r'\d+\sPlayers', regex=True)]

  # List of columns to be removed
  columns_to_remove = ['Total_Cmp', 'Total_Att', 'Total_Cmp%', 'Ast', 'xAG', 'PrgP', 'Att', 'Tackles_Tkl', 'Blocks_Blocks', 'Touches_Touches', 'Touches_Live', 'Crs']

  # Dropping columns from the df_merged DataFrame
  dataframe = dataframe.drop(columns=columns_to_remove, errors='ignore')

  # List of strings to be checked
  special_strings = ['Carries_Carries', 'Receiving', 'Performance', 'Expected', 'SCA']

  # Renaming columns from the 'special_strings' variable
  new_columns = [extract_last_word(col) if any(string in col for string in special_strings) else col for col in dataframe.columns]
  dataframe.columns = new_columns

  # Extracting country codes from 'Nation' column
  dataframe['Nation'] = dataframe['Nation'].str.split().str.get(1)

  # Converting the '#' column values to integers
  dataframe['#'] = dataframe['#'].astype(int)

  # Splitting the 'Age' column into two using the hyphen as a separator
  dataframe[['Age_Years', 'Age_Days']] = dataframe['Age'].str.split('-', expand=True)

  # Converting the new columns to numeric, handling errors with NaN
  dataframe['Age_Years'] = pd.to_numeric(dataframe['Age_Years'], errors='coerce')
  dataframe['Age_Days'] = pd.to_numeric(dataframe['Age_Days'], errors='coerce')

  # Removing the original 'Age' column
  dataframe = dataframe.drop(columns=['Age'])

  # Reordering the columns to maintain the original position of 'Age'
  age_index = dataframe.columns.get_loc('Nation')  # Finding the position of the 'Nation' column
  dataframe.insert(age_index + 1, 'Age_Years', dataframe.pop('Age_Years'))  # Inserting 'Age_Years' after 'Nation'
  dataframe.insert(age_index + 2, 'Age_Days', dataframe.pop('Age_Days'))  # Inserting 'Age_Days' after 'Age_Years'

  cleaned_df = dataframe

  return cleaned_df

In [ ]:
def enrich_df(dataframe, name1, name2):
  """Enriches the DataFrame with Match_ID, MD, and Squad columns.

  Args:
    dataframe: The pandas DataFrame to be enriched.
    name1: The name of the home team.
    name2: The name of the away team.

  Returns:
    The enriched pandas DataFrame.
  """
  # Adding the 'Match_ID' column in the first position
  if match_ID:  # Checks if match_ID is not empty (evaluates to True)
    dataframe.insert(0, 'Match_ID', match_ID)

  if MD:  # Checks if MD is not empty (evaluates to True)
    current_MD = ((match_ID - 1) // matches_per_MD) + 1
    dataframe.insert(0, 'MD', current_MD)

  if i == 0:
    dataframe.insert(dataframe.columns.get_loc('Player') + 1, 'Squad', name1)

  elif i == 1:
    dataframe.insert(dataframe.columns.get_loc('Player') + 1, 'Squad', name2)

  df_enriched = dataframe

  return df_enriched

## Data processing

In [ ]:
print(f"\nTotal number of URLs analysed: {len(urls)}")


Total number of URLs analysed: 18


In [ ]:
# Iterates through each match URL to extract and process data
for i, match_url in enumerate(urls):

  # Recording the start time
  start_time = time.time()

  # Read all HTML tables from the page using pandas
  dfs = pd.read_html(match_url)

  home_name, away_name = extract_squad_names(dfs)

  both_squads_dfs = extract_squads_dfs(dfs)

  # Iterates through each squad to extract and process data
  for i, current_squad_dfs in enumerate(both_squads_dfs):

    current_squad_dfs = clean_column_names(current_squad_dfs)

    all_squad_data = merge_dfs(current_squad_dfs, merge_keys)

    all_squad_data = clean_df(all_squad_data)

    all_squad_data = enrich_df(all_squad_data, home_name, away_name)

    # Concatenate the squad_data DataFrame to match_data
    all_data = pd.concat([all_data, all_squad_data], ignore_index=True)

  # Waits 10 seconds before the next request, which will extract data for the next squad
  # Used to avoid error 429 (too many requests)
  time.sleep(10)

  # Recording the end time
  end_time = time.time()
  # Calculating the elapsed time for processing each match's data
  elapsed_time = end_time - start_time
  print(f"Match {match_ID} ({home_name} vs {away_name}) successfully extracted and processed in {elapsed_time:.2f} seconds")

  # Updating match_ID
  if match_ID:
      match_ID += 1


Match 1 (Young Boys vs Aston Villa) successfully extracted and processed in 12.65 seconds
Match 2 (Juventus vs PSV Eindhoven) successfully extracted and processed in 10.64 seconds
Match 3 (Sporting CP vs Lille) successfully extracted and processed in 10.52 seconds
Match 4 (Real Madrid vs Stuttgart) successfully extracted and processed in 10.48 seconds
Match 5 (Milan vs Liverpool) successfully extracted and processed in 10.48 seconds
Match 6 (Bayern Munich vs Dinamo Zagreb) successfully extracted and processed in 10.45 seconds
Match 7 (Sparta Prague vs RB Salzburg) successfully extracted and processed in 10.46 seconds
Match 8 (Bologna vs Shakhtar) successfully extracted and processed in 10.46 seconds
Match 9 (Celtic vs Slovan Bratislava) successfully extracted and processed in 10.43 seconds
Match 10 (Manchester City vs Inter) successfully extracted and processed in 10.45 seconds
Match 11 (Club Brugge vs Dortmund) successfully extracted and processed in 10.45 seconds
Match 12 (Paris S-G 

In [ ]:
all_data

,MD,Match_ID,Player,Squad,#,Nation,Age_Years,Age_Days,Pos,Min,...,Off,Crs,TklW,PKwon,PKcon,OG,Recov,Aerial Duels_Won,Aerial Duels_Lost,Aerial Duels_Won%
0,1,1,Abdu Conté,Young Boys,22,POR,26,177,LB,28,...,0,0,0,0,0,0,1,0,0,NaN
1,1,1,Alan Virginius,Young Boys,21,FRA,21,258,"RM,FW",28,...,0,0,0,0,0,0,0,0,0,NaN
2,1,1,Cedric Itten,Young Boys,9,SUI,27,265,FW,8,...,0,0,0,0,0,0,0,1,0,100.0
3,1,1,Cheikh Tidiane Niasse,Young Boys,20,SEN,24,242,DM,45,...,0,0,0,0,0,0,2,1,0,100.0
4,1,1,Darian Males,Young Boys,39,SUI,23,137,LM,8,...,0,2,2,0,0,0,0,0,0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
564,1,18,Nicolas Seiwald,RB Leipzig,13,AUT,23,138,CM,31,...,0,0,1,0,0,0,1,0,1,0.0
565,1,18,Péter Gulácsi,RB Leipzig,1,HUN,34,136,GK,90,...,0,0,0,0,0,0,0,0,0,NaN
566,1,18,Willi Orban,RB Leipzig,4,HUN,31,321,CB,90,...,0,0,0,0,0,0,1,1,1,50.0
567,1,18,Xavi Simons,RB Leipzig,10,NED,21,151,LM,82,...,0,2,0,0,0,0,4,0,0,NaN


## Saving

In [ ]:
# Mount Google Drive
drive.mount('/content/drive')

all_data.to_csv(file_path, index=False)

Mounted at /content/drive
